# Fashion MNIST CNN (Notebook)

In [ ]:
%matplotlib inline
import warnings
warnings.filterwarnings("ignore")

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
import torchvision
import torchvision.transforms as transforms
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import time
import json
from pathlib import Path

#seed for reproducibility
torch.manual_seed(42)
np.random.seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

output_dir = Path("./outputs")
output_dir.mkdir(parents=True, exist_ok=True)

class_names = ['T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat',
               'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']

print("=" * 80)
print("FASHION MNIST CLASSIFICATION CNN")
print("=" * 80)

Using device: cuda
FASHION MNIST CLASSIFICATION CNN


## 1. DATA LOADING AND PREPROCESSING

In [ ]:
print("\n1. Loading and preprocessing data...")

# Data augmentation for training
train_transform = transforms.Compose([
    transforms.RandomRotation(5),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

# No augmentation for validation/test
test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

# Load Fashion MNIST dataset
train_dataset_full = torchvision.datasets.FashionMNIST(
    root='./data',
    train=True,
    download=True,
    transform=train_transform
)

# For validation set (no augmentation)
train_dataset_full_no_aug = torchvision.datasets.FashionMNIST(
    root='./data',
    train=True,
    download=True,
    transform=test_transform
)

test_dataset = torchvision.datasets.FashionMNIST(
    root='./data',
    train=False,
    download=True,
    transform=test_transform
)

# Split training data into train and validation sets (80-20 split)
train_size = int(0.8 * len(train_dataset_full))
val_size = len(train_dataset_full) - train_size

# Use augmented data for training
train_dataset, _ = random_split(
    train_dataset_full,
    [train_size, val_size],
    generator=torch.Generator().manual_seed(42)
)

# Use non-augmented data for validation
_, val_dataset = random_split(
    train_dataset_full_no_aug,
    [train_size, val_size],
    generator=torch.Generator().manual_seed(42)
)

print(f"Training samples: {len(train_dataset)}")
print(f"Validation samples: {len(val_dataset)}")
print(f"Test samples: {len(test_dataset)}")
print(f"Number of classes: {len(class_names)}")
print(f"Classes: {class_names}")

# Create data loaders
batch_size = 128
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)


1. Loading and preprocessing data...


100%|██████████| 26.4M/26.4M [00:02<00:00, 8.82MB/s]
100%|██████████| 29.5k/29.5k [00:00<00:00, 148kB/s]
100%|██████████| 4.42M/4.42M [00:01<00:00, 2.78MB/s]
100%|██████████| 5.15k/5.15k [00:00<00:00, 13.1MB/s]

Training samples: 48000
Validation samples: 12000
Test samples: 10000
Number of classes: 10
Classes: ['T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat', 'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']


## 2. BASELINE MODEL: K-NEAREST NEIGHBORS

In [ ]:
print("\n2. Training baseline model (K-Nearest Neighbors)...")

def prepare_data_for_sklearn(dataset, max_samples=10000):
    """Prepare data for sklearn models (flatten images)"""
    loader = DataLoader(dataset, batch_size=len(dataset), shuffle=False)
    images, labels = next(iter(loader))

    # Limit samples for faster training
    if len(images) > max_samples:
        indices = torch.randperm(len(images))[:max_samples]
        images = images[indices]
        labels = labels[indices]

    # Flatten images: (N, 1, 28, 28) -> (N, 784)
    images_flat = images.view(images.size(0), -1).numpy()
    labels_np = labels.numpy()

    return images_flat, labels_np

# Prepare data for baseline (use subset for speed)
print("Preparing data for KNN (using subset for computational efficiency)...")
# Create non-augmented version of train dataset for KNN
train_dataset_knn, _ = random_split(
    train_dataset_full_no_aug,
    [train_size, val_size],
    generator=torch.Generator().manual_seed(42)
)

X_train_base, y_train_base = prepare_data_for_sklearn(train_dataset_knn, max_samples=5000)
X_val_base, y_val_base = prepare_data_for_sklearn(val_dataset, max_samples=2000)
X_test_base, y_test_base = prepare_data_for_sklearn(test_dataset, max_samples=10000)

# Train KNN baseline
print("Training KNN classifier (k=3)...")
start_time = time.time()
knn_model = KNeighborsClassifier(n_neighbors=3, n_jobs=-1)
knn_model.fit(X_train_base, y_train_base)
train_time = time.time() - start_time

print(f"Training time: {train_time:.2f} seconds")

# Evaluate baseline
print("Evaluating baseline model...")
y_val_pred_base = knn_model.predict(X_val_base)
y_test_pred_base = knn_model.predict(X_test_base)

baseline_val_acc = accuracy_score(y_val_base, y_val_pred_base)
baseline_test_acc = accuracy_score(y_test_base, y_test_pred_base)

print(f"Baseline Validation Accuracy: {baseline_val_acc:.4f}")
print(f"Baseline Test Accuracy: {baseline_test_acc:.4f}")

# Save baseline results
baseline_results = {
    'model': 'K-Nearest Neighbors (k=3)',
    'validation_accuracy': float(baseline_val_acc),
    'test_accuracy': float(baseline_test_acc),
    'training_time_seconds': float(train_time)
}


2. Training baseline model (K-Nearest Neighbors)...
Preparing data for KNN (using subset for computational efficiency)...
Training KNN classifier (k=3)...
Training time: 0.00 seconds
Evaluating baseline model...
Baseline Validation Accuracy: 0.8050
Baseline Test Accuracy: 0.7947


## 3. CNN MODEL DEFINITION

In [ ]:
print("\n3. Defining CNN architecture...")

class FashionMNISTCNN(nn.Module):
    """
    Convolutional Neural Network for Fashion MNIST Classification

    Architecture:
    - Conv Block 1: 32 filters, 3x3 kernel, ReLU, MaxPool
    - Conv Block 2: 64 filters, 3x3 kernel, ReLU, MaxPool
    - Conv Block 3: 128 filters, 3x3 kernel, ReLU, MaxPool
    - Flatten
    - FC Layer 1: 256 neurons, ReLU, Dropout
    - FC Layer 2: 128 neurons, ReLU, Dropout
    - Output: 10 classes
    """

    def __init__(self, num_classes=10, dropout_rate=0.5):
        super(FashionMNISTCNN, self).__init__()

        # Convolutional layers
        self.conv_layers = nn.Sequential(
            # Conv Block 1: 28x28x1 -> 28x28x32 -> 14x14x32
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),

            # Conv Block 2: 14x14x32 -> 14x14x64 -> 7x7x64
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),

            # Conv Block 3: 7x7x64 -> 7x7x128 -> 3x3x128
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )

        # Calculate size after conv layers: 3x3x128 = 1152
        self.flatten_size = 3 * 3 * 128

        # Fully connected layers
        self.fc_layers = nn.Sequential(
            nn.Flatten(),
            nn.Linear(self.flatten_size, 256),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        x = self.conv_layers(x)
        x = self.fc_layers(x)
        return x


3. Defining CNN architecture...


## 4. TRAINING AND EVALUATION FUNCTIONS

In [ ]:
def train_epoch(model, loader, criterion, optimizer, device):
    """Train for one epoch"""
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)

        # Forward pass
        outputs = model(images)
        loss = criterion(outputs, labels)

        # Backward pass and optimization
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # Statistics
        running_loss += loss.item()
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    epoch_loss = running_loss / len(loader)
    epoch_acc = correct / total
    return epoch_loss, epoch_acc

def evaluate(model, loader, criterion, device):
    """Evaluate model on given data loader"""
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    all_predictions = []
    all_labels = []

    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            running_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

            all_predictions.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    avg_loss = running_loss / len(loader)
    accuracy = correct / total

    return avg_loss, accuracy, np.array(all_predictions), np.array(all_labels)

def train_model(model, train_loader, val_loader, criterion, optimizer,
                num_epochs, device, early_stopping_patience=7):
    """Train model with early stopping"""

    train_losses = []
    train_accs = []
    val_losses = []
    val_accs = []

    best_val_acc = 0.0
    best_model_state = None
    patience_counter = 0

    print(f"\nTraining for {num_epochs} epochs...")

    for epoch in range(num_epochs):
        # Train
        train_loss, train_acc = train_epoch(model, train_loader, criterion,
                                           optimizer, device)

        # Validate
        val_loss, val_acc, _, _ = evaluate(model, val_loader, criterion, device)

        # Store
        train_losses.append(train_loss)
        train_accs.append(train_acc)
        val_losses.append(val_loss)
        val_accs.append(val_acc)

        # Print progress
        print(f"Epoch [{epoch+1}/{num_epochs}] "
              f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f} | "
              f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}")

        # Early stopping , model checkpointing
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_model_state = model.state_dict().copy()
            patience_counter = 0
            print(f"   New best validation accuracy: {best_val_acc:.4f}")
        else:
            patience_counter += 1
            if patience_counter >= early_stopping_patience:
                print(f"\nEarly stopping triggered after {epoch+1} epochs")
                break

    # restore best model
    if best_model_state is not None:
        model.load_state_dict(best_model_state)
        print(f"\nRestored best model with validation accuracy: {best_val_acc:.4f}")

    history = {
        'train_losses': train_losses,
        'train_accs': train_accs,
        'val_losses': val_losses,
        'val_accs': val_accs,
        'best_val_acc': best_val_acc
    }

    return model, history

## 5. HYPERPARAMETER TUNING

In [ ]:
print("\n4. Hyperparameter tuning...")

# define hyperparameter configurations to try
configs = [
    {'dropout': 0.5, 'lr': 0.001, 'name': 'Config 1 (baseline)'},
    {'dropout': 0.3, 'lr': 0.001, 'name': 'Config 2 (less dropout)'},
    {'dropout': 0.5, 'lr': 0.0005, 'name': 'Config 3 (lower LR)'},
    {'dropout': 0.4, 'lr': 0.002, 'name': 'Config 4 (higher LR)'},
]

best_config = None
best_val_acc = 0.0
all_config_results = []

for i, config in enumerate(configs):
    print(f"\n{'='*60}")
    print(f"Testing {config['name']}: dropout={config['dropout']}, lr={config['lr']}")
    print(f"{'='*60}")

    # Initialize model
    model = FashionMNISTCNN(dropout_rate=config['dropout']).to(device)

    # loss + optimizer
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=config['lr'])

    # Train model
    model, history = train_model(
        model, train_loader, val_loader, criterion, optimizer,
        num_epochs=20, device=device, early_stopping_patience=5
    )

    # Store results
    config_result = {
        'config_name': config['name'],
        'config': config,
        'best_val_acc': history['best_val_acc'],
        'final_train_acc': history['train_accs'][-1],
    }
    all_config_results.append(config_result)

    print(f"\n{config['name']} Results:")
    print(f"  Best Validation Accuracy: {history['best_val_acc']:.4f}")

    # Track best config
    if history['best_val_acc'] > best_val_acc:
        best_val_acc = history['best_val_acc']
        best_config = config
        print(f"  New best configuration!")

print(f"\n{'='*60}")
print("Hyperparameter Tuning Complete")
print(f"{'='*60}")
print(f"Best Configuration: {best_config['name']}")
print(f"Best Validation Accuracy: {best_val_acc:.4f}")
print(f"Configuration: {best_config}")


4. Hyperparameter tuning...

Testing Config 1 (baseline): dropout=0.5, lr=0.001

Training for 20 epochs...
Epoch [1/20] Train Loss: 0.7695, Train Acc: 0.7146 | Val Loss: 0.4557, Val Acc: 0.8253
   New best validation accuracy: 0.8253
Epoch [2/20] Train Loss: 0.4930, Train Acc: 0.8217 | Val Loss: 0.3701, Val Acc: 0.8636
   New best validation accuracy: 0.8636
Epoch [3/20] Train Loss: 0.4312, Train Acc: 0.8459 | Val Loss: 0.3418, Val Acc: 0.8761
   New best validation accuracy: 0.8761
Epoch [4/20] Train Loss: 0.3910, Train Acc: 0.8618 | Val Loss: 0.3401, Val Acc: 0.8734
Epoch [5/20] Train Loss: 0.3659, Train Acc: 0.8704 | Val Loss: 0.2838, Val Acc: 0.8930
   New best validation accuracy: 0.8930
Epoch [6/20] Train Loss: 0.3549, Train Acc: 0.8757 | Val Loss: 0.2824, Val Acc: 0.8979
   New best validation accuracy: 0.8979
Epoch [7/20] Train Loss: 0.3333, Train Acc: 0.8830 | Val Loss: 0.2712, Val Acc: 0.8992
   New best validation accuracy: 0.8992
Epoch [8/20] Train Loss: 0.3225, Train Acc:

## 6. FINAL MODEL TRAINING WITH BEST HYPERPARAMETERS

In [ ]:
print(f"\n5. Training final model with best hyperparameters...")

# Initialize final model
final_model = FashionMNISTCNN(dropout_rate=best_config['dropout']).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(final_model.parameters(), lr=best_config['lr'])

# Train final model with more epochs
final_model, final_history = train_model(
    final_model, train_loader, val_loader, criterion, optimizer,
    num_epochs=30, device=device, early_stopping_patience=7
)


5. Training final model with best hyperparameters...

Training for 30 epochs...
Epoch [1/30] Train Loss: 0.6406, Train Acc: 0.7626 | Val Loss: 0.3854, Val Acc: 0.8606
   New best validation accuracy: 0.8606
Epoch [2/30] Train Loss: 0.4204, Train Acc: 0.8460 | Val Loss: 0.3748, Val Acc: 0.8664
   New best validation accuracy: 0.8664
Epoch [3/30] Train Loss: 0.3661, Train Acc: 0.8655 | Val Loss: 0.3179, Val Acc: 0.8853
   New best validation accuracy: 0.8853
Epoch [4/30] Train Loss: 0.3364, Train Acc: 0.8763 | Val Loss: 0.2925, Val Acc: 0.8965
   New best validation accuracy: 0.8965
Epoch [5/30] Train Loss: 0.3160, Train Acc: 0.8851 | Val Loss: 0.2721, Val Acc: 0.8988
   New best validation accuracy: 0.8988
Epoch [6/30] Train Loss: 0.3011, Train Acc: 0.8910 | Val Loss: 0.2915, Val Acc: 0.8972
Epoch [7/30] Train Loss: 0.2898, Train Acc: 0.8940 | Val Loss: 0.2490, Val Acc: 0.9077
   New best validation accuracy: 0.9077
Epoch [8/30] Train Loss: 0.2771, Train Acc: 0.8982 | Val Loss: 0.2490,

## 7. FINAL EVALUATION ON TEST SET

In [ ]:
print("\n6. Final evaluation on test set...")

test_loss, test_acc, test_predictions, test_labels = evaluate(
    final_model, test_loader, criterion, device
)

print(f"\nFinal Test Accuracy: {test_acc:.4f}")
print(f"Final Test Loss: {test_loss:.4f}")

# Detailed classification report
print("\nClassification Report:")
print(classification_report(test_labels, test_predictions,
                          target_names=class_names, digits=4))


6. Final evaluation on test set...

Final Test Accuracy: 0.9230
Final Test Loss: 0.2091

Classification Report:
              precision    recall  f1-score   support

 T-shirt/top     0.8939    0.8340    0.8629      1000
     Trouser     0.9970    0.9870    0.9920      1000
    Pullover     0.8577    0.9100    0.8831      1000
       Dress     0.9000    0.9360    0.9176      1000
        Coat     0.9104    0.8430    0.8754      1000
      Sandal     0.9879    0.9820    0.9850      1000
       Shirt     0.7617    0.7990    0.7799      1000
     Sneaker     0.9663    0.9760    0.9711      1000
         Bag     0.9890    0.9890    0.9890      1000
  Ankle boot     0.9769    0.9740    0.9755      1000

    accuracy                         0.9230     10000
   macro avg     0.9241    0.9230    0.9231     10000
weighted avg     0.9241    0.9230    0.9231     10000



## 8. VISUALIZATION AND ANALYSIS

In [ ]:
print("\n7. Generating visualizations and analysis...")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss plot
axes[0].plot(final_history['train_losses'], label='Train Loss', linewidth=2)
axes[0].plot(final_history['val_losses'], label='Validation Loss', linewidth=2)
axes[0].set_xlabel('Epoch', fontsize=12)
axes[0].set_ylabel('Loss', fontsize=12)
axes[0].set_title('Training and Validation Loss (CNN)', fontsize=14, fontweight='bold')
axes[0].legend(fontsize=11)
axes[0].grid(True, alpha=0.3)

# Accuracy plot
axes[1].plot(final_history['train_accs'], label='Train Accuracy', linewidth=2)
axes[1].plot(final_history['val_accs'], label='Validation Accuracy', linewidth=2)
axes[1].set_xlabel('Epoch', fontsize=12)
axes[1].set_ylabel('Accuracy', fontsize=12)
axes[1].set_title('Training and Validation Accuracy (CNN)', fontsize=14, fontweight='bold')
axes[1].legend(fontsize=11)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(output_dir / 'training_history.png', dpi=300, bbox_inches='tight')
print("Saved: training_history.png")
plt.close()

# Confusion Matrix
cm = confusion_matrix(test_labels, test_predictions)
plt.figure(figsize=(12, 10))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names,
            cbar_kws={'label': 'Count'})
plt.xlabel('Predicted Label', fontsize=12)
plt.ylabel('True Label', fontsize=12)
plt.title('Confusion Matrix - Test Set (CNN)', fontsize=14, fontweight='bold')
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig(output_dir / 'confusion_matrix.png', dpi=300, bbox_inches='tight')
print("Saved: confusion_matrix.png")
plt.close()

# Per-class accuracy
per_class_acc = cm.diagonal() / cm.sum(axis=1)
plt.figure(figsize=(12, 6))
bars = plt.bar(range(len(class_names)), per_class_acc, color='steelblue', alpha=0.8)
plt.xlabel('Class', fontsize=12)
plt.ylabel('Accuracy', fontsize=12)
plt.title('Per-Class Accuracy on Test Set (CNN)', fontsize=14, fontweight='bold')
plt.xticks(range(len(class_names)), class_names, rotation=45, ha='right')
plt.ylim([0, 1])
plt.grid(axis='y', alpha=0.3)


for i, (bar, acc) in enumerate(zip(bars, per_class_acc)):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
             f'{acc:.3f}', ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.savefig(output_dir / 'per_class_accuracy.png', dpi=300, bbox_inches='tight')
print("Saved: per_class_accuracy.png")
plt.close()

# Hyperparameter comparison
config_names = [r['config_name'] for r in all_config_results]
config_accs = [r['best_val_acc'] for r in all_config_results]

plt.figure(figsize=(10, 6))
bars = plt.bar(config_names, config_accs, color='coral', alpha=0.8)
plt.xlabel('Configuration', fontsize=12)
plt.ylabel('Validation Accuracy', fontsize=12)
plt.title('Hyperparameter Configuration Comparison (CNN)', fontsize=14, fontweight='bold')
plt.ylim([min(config_accs) - 0.01, max(config_accs) + 0.01])
plt.xticks(rotation=15, ha='right')
plt.grid(axis='y', alpha=0.3)

# best config
best_idx = config_accs.index(max(config_accs))
bars[best_idx].set_color('green')
bars[best_idx].set_alpha(0.9)

# Add value labels
for bar, acc in zip(bars, config_accs):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001,
             f'{acc:.4f}', ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.savefig(output_dir / 'hyperparameter_comparison.png', dpi=300, bbox_inches='tight')
print("Saved: hyperparameter_comparison.png")
plt.close()

# Sample predictions visualization
def show_sample_predictions(model, test_loader, device, num_samples=10):
    """Show sample predictions with correct and incorrect classifications"""
    model.eval()


    images, labels = next(iter(test_loader))
    images, labels = images.to(device), labels.to(device)

    with torch.no_grad():
        outputs = model(images)
        _, predicted = torch.max(outputs, 1)

    # Find correct and incorrect predictions
    correct_mask = (predicted == labels)
    incorrect_mask = ~correct_mask


    correct_indices = torch.where(correct_mask)[0][:num_samples//2]
    incorrect_indices = torch.where(incorrect_mask)[0][:num_samples//2]

    sample_indices = torch.cat([correct_indices, incorrect_indices])

    # Plot
    fig, axes = plt.subplots(2, num_samples//2, figsize=(15, 6))
    fig.suptitle('Sample Predictions: Correct (Top) vs Incorrect (Bottom) - CNN',
                 fontsize=14, fontweight='bold')

    for i, idx in enumerate(sample_indices):
        row = i // (num_samples//2)
        col = i % (num_samples//2)

        img = images[idx].cpu().squeeze()
        true_label = class_names[labels[idx]]
        pred_label = class_names[predicted[idx]]

        axes[row, col].imshow(img, cmap='gray')
        axes[row, col].axis('off')

        color = 'green' if predicted[idx] == labels[idx] else 'red'
        axes[row, col].set_title(f'True: {true_label}\nPred: {pred_label}',
                                 fontsize=9, color=color)

    plt.tight_layout()
    plt.savefig(output_dir / 'sample_predictions.png', dpi=300, bbox_inches='tight')
    print("Saved: sample_predictions.png")
    plt.close()

show_sample_predictions(final_model, test_loader, device, num_samples=10)


7. Generating visualizations and analysis...
Saved: training_history.png
Saved: confusion_matrix.png
Saved: per_class_accuracy.png
Saved: hyperparameter_comparison.png
Saved: sample_predictions.png


## 9. SAVE RESULTS AND MODEL

In [ ]:
print("\n8. Saving results...")
# test predictions
np.save(output_dir / 'test_predictions.npy', test_predictions)
np.save(output_dir / 'test_labels.npy', test_labels)
print("Saved: test_predictions.npy and test_labels.npy")

# model
torch.save(final_model.state_dict(), output_dir / 'best_model.pth')
print("Saved: best_model.pth")


final_results = {
    'dataset': 'Fashion MNIST',
    'num_classes': len(class_names),
    'class_names': class_names,
    'data_split': {
        'train': len(train_dataset),
        'validation': len(val_dataset),
        'test': len(test_dataset)
    },
    'baseline': baseline_results,
    'cnn': {
        'architecture': {
            'type': 'Convolutional Neural Network',
            'conv_blocks': 3,
            'conv_filters': [32, 64, 128],
            'fc_layers': [256, 128],
            'dropout_rate': best_config['dropout'],
            'learning_rate': best_config['lr'],
            'optimizer': 'Adam',
            'loss_function': 'CrossEntropyLoss',
            'data_augmentation': 'RandomRotation(5) + RandomAffine(translate=0.1)',
            'batch_normalization': True
        },
        'hyperparameter_tuning': {
            'configurations_tested': len(configs),
            'best_config': best_config,
            'all_results': all_config_results
        },
        'training': {
            'epochs_trained': len(final_history['train_accs']),
            'best_validation_accuracy': float(final_history['best_val_acc']),
            'final_train_accuracy': float(final_history['train_accs'][-1]),
            'final_val_accuracy': float(final_history['val_accs'][-1])
        },
        'test_performance': {
            'test_accuracy': float(test_acc),
            'test_loss': float(test_loss),
            'per_class_accuracy': {
                class_names[i]: float(per_class_acc[i])
                for i in range(len(class_names))
            }
        }
    },
    'comparison': {
        'baseline_test_accuracy': float(baseline_test_acc),
        'cnn_test_accuracy': float(test_acc),
        'improvement': float(test_acc - baseline_test_acc),
        'relative_improvement_percent': float((test_acc - baseline_test_acc) / baseline_test_acc * 100)
    }
}

# Save  JSON
with open(output_dir / 'results.json', 'w') as f:
    json.dump(final_results, f, indent=2)
print("Saved: results.json")


8. Saving results...
Saved: test_predictions.npy and test_labels.npy
Saved: best_model.pth
Saved: results.json


## 10. SUMMARY

In [ ]:
print("\n" + "="*80)
print("FINAL RESULTS SUMMARY")
print("="*80)
print(f"\nDataset: Fashion MNIST")
print(f"Task: Multi-class classification (10 classes)")
print(f"\nData Split:")
print(f"  Training:   {len(train_dataset):,} samples")
print(f"  Validation: {len(val_dataset):,} samples")
print(f"  Test:       {len(test_dataset):,} samples")
print(f"\nBaseline Model (KNN):")
print(f"  Test Accuracy: {baseline_test_acc:.4f} ({baseline_test_acc*100:.2f}%)")
print(f"\nCNN Model:")
print(f"  Architecture: 3 Conv blocks [32, 64, 128] + 2 FC layers [256, 128]")
print(f"  Test Accuracy: {test_acc:.4f} ({test_acc*100:.2f}%)")
print(f"  Test Loss: {test_loss:.4f}")
print(f"\nImprovement over Baseline:")
print(f"  Absolute: +{(test_acc - baseline_test_acc):.4f}")
print(f"  Relative: +{((test_acc - baseline_test_acc) / baseline_test_acc * 100):.2f}%")
print(f"\nBest performing classes:")
sorted_classes = sorted(zip(class_names, per_class_acc), key=lambda x: x[1], reverse=True)
for i, (cls, acc) in enumerate(sorted_classes[:3]):
    print(f"  {i+1}. {cls}: {acc:.4f}")
print(f"\nWorst performing classes:")
for i, (cls, acc) in enumerate(sorted_classes[-3:]):
    print(f"  {i+1}. {cls}: {acc:.4f}")
print("\n" + "="*80)
print("All outputs saved to:", output_dir)
print("="*80)


FINAL RESULTS SUMMARY

Dataset: Fashion MNIST
Task: Multi-class classification (10 classes)

Data Split:
  Training:   48,000 samples
  Validation: 12,000 samples
  Test:       10,000 samples

Baseline Model (KNN):
  Test Accuracy: 0.7947 (79.47%)

CNN Model:
  Architecture: 3 Conv blocks [32, 64, 128] + 2 FC layers [256, 128]
  Test Accuracy: 0.9230 (92.30%)
  Test Loss: 0.2091

Improvement over Baseline:
  Absolute: +0.1283
  Relative: +16.14%

Best performing classes:
  1. Bag: 0.9890
  2. Trouser: 0.9870
  3. Sandal: 0.9820

Worst performing classes:
  1. Coat: 0.8430
  2. T-shirt/top: 0.8340
  3. Shirt: 0.7990

All outputs saved to: outputs
